# Diferentes Tipos de Arquivos no Spark

## Principais Formatos Usados em Big Data:

### 1. **Parquet** (Formato Colunar)
* Formato colunar otimizado para leitura
* Compressão eficiente
* Melhor performance para queries analíticas
* Suporta schemas complexos

### 2. **Delta Lake** (Formato Transacional)
* Formato nativo do Databricks
* ACID transactions
* Time travel (versionamento)
* Schema evolution
* Upserts e deletes eficientes

### 3. **JSON** (Formato Semi-Estruturado)
* Formato legível por humanos
* Flexível para dados semi-estruturados
* Maior tamanho de arquivo
* Mais lento para processar

### 4. **CSV** (Formato Texto)
* Formato simples e universal
* Legível por humanos
* Sem suporte nativo a tipos complexos
* Requer definição de schema

### 5. **ORC** (Optimized Row Columnar)
* Formato colunar otimizado
* Compressão eficiente
* Popular no ecossistema Hive
* Suporta ACID transactions

### 6. **Avro** (Formato de Linha)
* Formato binário compacto
* Schema incluído no arquivo
* Otimizado para escrita
* Bom para streaming e serialização

In [0]:
df_categories = spark.read.csv("/Volumes/curso_databricks/aula/aula_volume/Bike Store/categories.csv", header=True, inferSchema=True)
df_categories.show()

## PARTE 1: Salvando Dados em Diferentes Formatos
Vamos salvar o DataFrame em todos os formatos principais

In [0]:
# 1. PARQUET - Formato Colunar Otimizado
parquet_path = "/Volumes/curso_databricks/aula/aula_volume/Bike Store/formats/parquet"
df_categories.write.mode("overwrite").parquet(parquet_path)


In [0]:
# 2. DELTA - Formato Transacional do Databricks
delta_path = "/Volumes/curso_databricks/aula/aula_volume/Bike Store/formats/delta"
df_categories.write.mode("overwrite").format("delta").save(delta_path)


In [0]:
# 3. JSON - Formato Semi-Estruturado
json_path = "/Volumes/curso_databricks/aula/aula_volume/Bike Store/formats/json"
df_categories.write.mode("overwrite").json(json_path)

In [0]:
# 4. CSV - Formato Texto Simples
csv_path = "/Volumes/curso_databricks/aula/aula_volume/Bike Store/formats/csv"
df_categories.write.mode("overwrite").option("header", "true").csv(csv_path)

In [0]:
# 5. ORC - Optimized Row Columnar
orc_path = "/Volumes/curso_databricks/aula/aula_volume/Bike Store/formats/orc"
df_categories.write.mode("overwrite").orc(orc_path)

In [0]:
# 6. AVRO - Formato de Linha Compacto
avro_path = "/Volumes/curso_databricks/aula/aula_volume/Bike Store/formats/avro"
df_categories.write.mode("overwrite").format("avro").save(avro_path)

##  PARTE 2: Lendo Dados de Diferentes Formatos
Vamos ler os dados salvos em cada formato e exibir os resultados

In [0]:
# 1. LER PARQUET
df_parquet = spark.read.parquet(parquet_path)
df_parquet.show()

In [0]:
# 2. LER DELTA
df_delta = spark.read.format("delta").load(delta_path)
df_delta.show()

In [0]:
# 3. LER JSON
df_json = spark.read.json(json_path)
df_json.show()

In [0]:
# 4. LER CSV
df_csv = spark.read.option("header", "true").option("inferSchema", "true").csv(csv_path)
df_csv.show()

In [0]:
# 5. LER ORC
df_orc = spark.read.orc(orc_path)
df_orc.show()


In [0]:
# 6. LER AVRO
df_avro = spark.read.format("avro").load(avro_path)
df_avro.show()


## PARTE 3: Comparação de Tamanhos e Performance
Vamos comparar o tamanho dos arquivos gerados em cada formato

In [0]:
import os

def get_directory_size(path):
    """Calcula o tamanho total de um diretório em bytes"""
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path.replace("/Volumes/", "/Volumes/")):
        for filename in filenames:
            filepath = os.path.join(dirpath, filename)
            try:
                total_size += os.path.getsize(filepath)
            except:
                pass
    return total_size

def bytes_to_readable(bytes_size):
    """Converte bytes para formato legível"""
    for unit in ['B', 'KB', 'MB', 'GB']:
        if bytes_size < 1024.0:
            return f"{bytes_size:.2f} {unit}"
        bytes_size /= 1024.0
    return f"{bytes_size:.2f} TB"

formats_paths = {
    'Parquet': parquet_path,
    'Delta': delta_path,
    'JSON': json_path,
    'CSV': csv_path,
    'ORC': orc_path,
    'Avro': avro_path
}

print("\n" + "=" * 70)
print("COMPARAÇÃO DE TAMANHO DOS ARQUIVOS")
print("=" * 70)

for format_name, path in formats_paths.items():
    # Remove o prefixo /Volumes/ e adiciona /Volumes/ para o caminho real
    real_path = path.replace("/Volumes/", "/Volumes/")
    try:
        size = get_directory_size(real_path)
        readable_size = bytes_to_readable(size)
        print(f"{format_name:12} -> {readable_size:>12}")
    except Exception as e:
        print(f"{format_name:12} -> Erro ao calcular tamanho")

print("=" * 70)

##  PARTE 4: Quando Usar Cada Formato?

###  Recomendações de Uso:

| Formato | Melhor Para | Evitar Quando |
|---------|-------------|---------------|
| **Parquet** | Data lakes, queries analíticas, leitura intensiva | Dados que mudam frequentemente |
| **Delta** | Pipelines de dados, operações Schema, data lakes transacionais  versionamento e controle CICD | Sistemas legados sem suporte |
| **JSON** | APIs, dados semi-estruturados, interoperabilidade | Grandes volumes de dados |
| **CSV** | Export/import simples, Excel, ferramentas legadas | Dados com schemas complexos |
| **ORC** | Ecossistema Hive, compressão máxima | Quando não usa Hive |
| **Avro** | Streaming, Kafka, schema evolution | Queries analíticas |

###  Boas Práticas:

* **Para Data Engineering no Databricks**: Use **Delta** como padrão
* **Para Analytics/BI**: Use **Parquet** ou **Delta**
* **Para Integrações Externas**: Use **JSON** ou **CSV**
* **Para Streaming**: Use **Avro** ou **Delta**
* **Para Compressão Máxima**: Use **ORC** ou **Parquet com compressão**

###  Opções Avançadas:

```python
# Parquet com compressão
df.write.option("compression", "snappy").parquet(path)

# Delta com otimização
df.write.format("delta").option("optimizeWrite", "true").save(path)

# Particionamento
df.write.partitionBy("coluna").parquet(path)

# Modo de escrita
df.write.mode("overwrite")  # append, overwrite, ignore, error
```